# 05 — Catalyst Optimizer

**Concept:** Catalyst is the query optimizer inside Spark SQL. It transforms your Python/SQL expression into the most efficient physical execution plan through four stages — and knowing those stages is how you debug plans, understand why predicate pushdown fired (or didn't), and explain Cost-Based Optimization (**CBO**) vs Rule-Based Optimization (**RBO**) to an interviewer.

**Core interview question:** *"What is predicate pushdown and why does it matter?"*

## What you will observe
1. The **four plan stages** — parsed → analyzed → optimized logical → physical — visible in `explain(mode='extended')`.
2. **Predicate pushdown (RBO):** filter moves below a `Project` or `Join` in the optimized plan.
3. **Column pruning (RBO):** only referenced columns appear in `LocalTableScan` — unused columns are dropped before any work.
4. **Constant folding & boolean simplification (RBO):** `1 + 1` becomes `2` at plan time; `true AND x` becomes `x`.
5. **Cost-based optimization (CBO):** how `ANALYZE TABLE` changes the join order Catalyst chooses.
6. **WholeStageCodegen (Tungsten):** which operators are fused into a single JVM bytecode loop and which break the fusion boundary.
7. **Where predicate pushdown does NOT fire** — and how to diagnose it.

## The three optimization layers in Spark — and how they relate

Spark has three distinct optimization systems. They operate at different times and on different problems. Confusing them is one of the most common gaps in interview answers.

```
Your query
    │
    ▼
┌─────────────────────────────────────────────────────────────┐
│  CATALYST  (plan-time optimizer)                            │
│  Runs once, before execution, on the logical plan.          │
│  Output: a physical plan with operator choices baked in.    │
│                                                             │
│  ├── RBO: predicate pushdown, column pruning, constant fold │
│  └── CBO: join order, join strategy (needs ANALYZE TABLE)   │
└─────────────────────────────┬───────────────────────────────┘
                              │  physical plan
                              ▼
┌─────────────────────────────────────────────────────────────┐
│  AQE  (runtime re-optimizer — Catalyst's adaptive layer)    │
│  Runs at shuffle boundaries, after each stage completes.    │
│  Has access to actual shuffle file sizes — ground truth.    │
│                                                             │
│  ├── Coalesce shuffle partitions (200 → actual N needed)    │
│  ├── Convert SMJ → BHJ if build side turns out to be small  │
│  └── Split skewed shuffle partitions into sub-tasks         │
└─────────────────────────────┬───────────────────────────────┘
                              │  (possibly rewritten) plan
                              ▼
┌─────────────────────────────────────────────────────────────┐
│  TUNGSTEN  (execution engine)                               │
│  Not an optimizer — executes whatever plan Catalyst/AQE     │
│  produced, but does so as fast as possible.                 │
│                                                             │
│  ├── UnsafeRow: compact binary format, off-heap, no GC      │
│  ├── Cache-aware sort/hash algorithms                       │
│  └── WholeStageCodegen: JIT-compiles operator pipelines     │
│      into a single tight loop (no virtual dispatch)         │
└─────────────────────────────────────────────────────────────┘
```

### The key distinctions

| | Catalyst (static) | AQE | Tungsten |
|---|---|---|---|
| **When** | Plan time — before execution | Runtime — after each shuffle stage | Execution — always running |
| **Inputs** | Logical plan + catalog statistics | Actual shuffle file sizes | Whatever physical plan exists |
| **What it changes** | The plan structure (filter position, join strategy, join order) | The plan structure (mid-execution rewrites) | Nothing in the plan — only *how fast* operators run |
| **Requires stats?** | CBO yes; RBO no | No — reads actual file sizes | No |
| **Visible in `explain()`?** | Yes — Optimized Logical Plan | Yes — `isFinalPlan=true` post-execution plan | Yes — `WholeStageCodegen` boxes, `UnsafeRow` in codegen output |

### One-sentence summaries for interviews
- **Catalyst** — *"the query planner that transforms your logical intent into an efficient physical plan before any data moves."*
- **AQE** — *"Catalyst running again at runtime, after each shuffle, with actual data statistics instead of estimates."*
- **Tungsten** — *"the execution engine that makes the chosen plan run as fast as possible — compact memory layout, CPU-cache-friendly algorithms, and JIT-compiled operator pipelines."*

**This notebook focuses on static Catalyst.** AQE gets its own deep dive in notebook 06.

## The four Catalyst plan stages

```
User code (Python/SQL)
        │
        ▼
  Parsed Logical Plan      ← AST: unresolved column names and functions
        │   (Analyzer: resolve names against catalog)
        ▼
  Analyzed Logical Plan    ← all names resolved; types checked
        │   (Optimizer: apply RBO and CBO rule batches)
        ▼
  Optimized Logical Plan   ← filters pushed, columns pruned, constants folded
        │   (Planner: select physical operators)
        ▼
  Physical Plan            ← BroadcastHashJoin vs SortMergeJoin; Exchange nodes; WholeStageCodegen boundaries
```

| Stage | Key question answered | What changes |
|---|---|---|
| Parsed | Can we parse the SQL/Python? | Unresolved attribute names, function calls |
| Analyzed | Are all names valid? | Attribute IDs assigned; types resolved; missing columns caught |
| Optimized logical | Can we compute the same result cheaper? | Filters pushed down; columns pruned; subqueries flattened; join order (if CBO) |
| Physical | How do we actually execute this? | Join strategy; sort order; AQE wrapping; `Exchange` nodes; codegen boundaries |

In [1]:
import sys
import os
from pathlib import Path

sys.path.append(str(Path().absolute().parent))

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ.setdefault("HADOOP_HOME", r"C:\hadoop")
os.environ["PATH"] = r"C:\hadoop\bin" + os.pathsep + os.environ.get("PATH", "")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
import pandas as pd
import random

warehouse_path = str(Path().absolute().parent / "data" / "warehouse")
os.makedirs(warehouse_path, exist_ok=True)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("05-catalyst-optimizer")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.cbo.enabled", "true")
    .config("spark.sql.statistics.histogram.enabled", "true")
    .config("spark.sql.warehouse.dir", warehouse_path)
    .config("spark.python.worker.reuse", "true")
    
    # Java 17 blocks sun.misc.Unsafe access needed by Arrow; disable to avoid
    # the cloudpickle recursion that occurs in the Arrow fallback serializer.
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version}  —  UI: http://localhost:4040")
print(f"Java              : {os.environ['JAVA_HOME']}")
print(f"CBO enabled       : {spark.conf.get('spark.sql.cbo.enabled')}")
print(f"AQE enabled       : {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"autoBroadcast     : {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')}")
print(f"HADOOP_HOME       : {os.environ.get('HADOOP_HOME', 'NOT SET')}")

Spark 3.5.8  —  UI: http://localhost:4040
Java              : C:\Program Files\Java\jdk-17.0.19
CBO enabled       : true
AQE enabled       : true
autoBroadcast     : 10485760b
HADOOP_HOME       : C:\hadoop


## Build sample datasets

Same tables as notebook 04 so the plans are comparable.

| Table | Rows | Used for |
|---|---|---|
| `products` | 20 | column pruning & predicate pushdown demos |
| `customers` | 200 | CBO join-order demo |
| `orders` | 10 000 | large fact table |

In [2]:
random.seed(42)

products_schema = StructType([
    StructField("product_id",   IntegerType(), False),
    StructField("product_name", StringType(),  True),
    StructField("category",     StringType(),  True),
    StructField("price",        DoubleType(),  True),
    StructField("stock",        IntegerType(), True),
    StructField("description",  StringType(),  True),
    StructField("sku",          StringType(),  True),
])
products_data = [
    (i, f"product_{i}", f"cat_{(i % 4) + 1}", round(random.uniform(5.0, 500.0), 2),
     random.randint(0, 1000), f"desc_{i}", f"sku_{i:04d}")
    for i in range(1, 21)
]
products = spark.createDataFrame(products_data, schema=products_schema)

customers_schema = StructType([
    StructField("customer_id",   IntegerType(), False),
    StructField("customer_name", StringType(),  True),
    StructField("country_id",    IntegerType(), True),
])
customers = spark.createDataFrame(
    [(i, f"customer_{i}", (i % 5) + 1) for i in range(1, 201)],
    schema=customers_schema,
)

orders_schema = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id",  IntegerType(), True),
    StructField("amount",      DoubleType(),  True),
])
orders_rows = [
    (i, random.randint(1, 200), random.randint(1, 20),
     round(random.uniform(10.0, 2000.0), 2))
    for i in range(1, 10001)
]
orders = spark.createDataFrame(orders_rows, schema=orders_schema)

print(f"products  : {products.count()} rows, {len(products.columns)} columns")
print(f"customers : {customers.count()} rows")
print(f"orders    : {orders.count()} rows")

products  : 20 rows, 7 columns
customers : 200 rows
orders    : 10000 rows


## 1. The four plan stages in `explain(mode='extended')`

`explain(mode='extended')` prints all four plan stages in one call. Read them in order:
1. **Parsed** — column names are unresolved strings.
2. **Analyzed** — every `'col` becomes `col#ID: type`.
3. **Optimized logical** — rules have fired: filters moved, columns dropped.
4. **Physical** — join strategies, `Exchange` nodes, codegen groups.

The most informative comparison is **Analyzed vs Optimized** — that gap is exactly what Catalyst's rule engine did.

In [3]:
# A query that will show all four stages clearly:
# filter + select on a wide table — predicate pushdown and column pruning will both fire.
query = (
    products
    .filter(F.col("price") > 100)
    .select("product_id", "product_name", "category", "price")
)

print("All four plan stages (explain mode='extended'):")
print("=" * 70)
query.explain(mode="extended")

All four plan stages (explain mode='extended'):
== Parsed Logical Plan ==
'Project ['product_id, 'product_name, 'category, 'price]
+- Filter (price#3 > cast(100 as double))
   +- LogicalRDD [product_id#0, product_name#1, category#2, price#3, stock#4, description#5, sku#6], false

== Analyzed Logical Plan ==
product_id: int, product_name: string, category: string, price: double
Project [product_id#0, product_name#1, category#2, price#3]
+- Filter (price#3 > cast(100 as double))
   +- LogicalRDD [product_id#0, product_name#1, category#2, price#3, stock#4, description#5, sku#6], false

== Optimized Logical Plan ==
Project [product_id#0, product_name#1, category#2, price#3]
+- Filter (isnotnull(price#3) AND (price#3 > 100.0))
   +- LogicalRDD [product_id#0, product_name#1, category#2, price#3, stock#4, description#5, sku#6], false

== Physical Plan ==
*(1) Project [product_id#0, product_name#1, category#2, price#3]
+- *(1) Filter (isnotnull(price#3) AND (price#3 > 100.0))
   +- *(1) Scan E

### What to observe in the four stages

**Parsed logical plan:**
- Column references appear as `'price`, `'product_id` — the tick prefix means *unresolved*.
- Function calls have no type information yet.

**Analyzed logical plan:**
- `'price` becomes `price#ID: double` — resolved against the schema, type-checked.
- The `Filter` still sits above the `Project` — Catalyst has not optimized yet, just resolved.

**Optimized logical plan:**
- The `Filter (price > 100)` moves **below** the `Project` (predicate pushdown). Rows are discarded before columns are selected.
- Only the four selected columns appear in the `LocalRelation` output — unused columns (`stock`, `description`, `sku`) are dropped (column pruning).
- Compare column lists between Analyzed and Optimized — the drop is visible.
- `LocalRelation` is the logical representation of in-memory driver data; `LocalTableScan` is its physical execution counterpart. When filters are applied to a `LocalRelation`, Catalyst evaluates them at optimization time and folds the result into the relation — which is why the filter node vanishes from the Optimized plan. This is a special case: it only happens for in-memory data. File-based (parquet) sources cannot do this, so their filters remain visible as `PushedFilters` on the `FileScan` node.

**Physical plan:**
- `Filter` is fused with the scan inside a `WholeStageCodegen` boundary.
- No `Exchange` node — this is a narrow pipeline (filter + project), one stage.

**Interview shorthand:** "The gap between Analyzed and Optimized is where Catalyst earns its keep."

## 2. Rule-based optimization (RBO)

RBO applies a fixed set of algebraic rewrite rules to the logical plan — **no statistics required**. Rules fire unconditionally whenever their pattern matches the plan tree.

The most important rules for interviews:

| Rule | What it does | Why it matters |
|---|---|---|
| **Predicate pushdown** | Moves `Filter` nodes as close to the data source as possible | Reduces rows before any computation or shuffle |
| **Column pruning** | Removes unused columns from `Project` nodes | Reduces bytes read, especially on columnar formats (Parquet) |
| **Constant folding** | Evaluates constant expressions at plan time | `1 + 1` → `2`; never recomputed per row |
| **Boolean simplification** | Rewrites tautologies/contradictions | `true AND x` → `x`; `x OR true` → `true` |
| **Null propagation** | Simplifies nullability reasoning | Avoids redundant null checks |
| **Subquery elimination** | Flattens correlated subqueries into joins | Avoids re-executing the subquery per outer row |

### 2a. Predicate pushdown

**The rule:** move a `Filter` node past a `Project` (select), past an `Aggregate`, or past one side of a `Join` — wherever the filtered column is available.

**Why it matters:**
- Fewer rows reaching expensive operators (joins, aggregations, shuffles).
- At the file reader level (Parquet), the filter can skip entire row groups — before Spark even constructs a partition (covered in depth in notebook 09).

**Where it does NOT fire:**
- Filter references a column computed by the `Project` (the column does not exist before the project).
- Filter contains a non-deterministic expression (e.g., `rand()`, `uuid()`).
- Filter crosses a `DISTINCT` or window boundary in ways that would change semantics.

> **Why we use Parquet here instead of in-memory data:**
> With a `LocalRelation` (data created via `createDataFrame`), Catalyst evaluates filters at plan time and folds them directly into the relation — the `Filter` node disappears entirely from the Optimized plan, making it impossible to observe pushdown. With a Parquet source, Catalyst cannot read the file at plan time, so the `Filter` node stays visible in the plan. You can clearly see it sitting above or below the `Project`, and confirm it appears as `PushedFilters` on the `FileScan` node in the physical plan.

In [4]:
parquet_dir = Path().absolute().parent / "data" / "products_pushdown"
parquet_dir.mkdir(parents=True, exist_ok=True)
products.toPandas().to_parquet(str(parquet_dir / "part-0.parquet"), index=False)

spark.sql("DROP TABLE IF EXISTS products_pushdown")
spark.sql(f"CREATE TABLE products_pushdown USING PARQUET LOCATION '{parquet_dir.as_uri()}'")
products_pq = spark.table("products_pushdown")

print("Written via pandas → registered as external table 'products_pushdown'")
print()
print("Sanity check — plan must show 'FileScan parquet', not 'LocalTableScan':")
products_pq.select("product_id", "price").explain(mode="formatted")

Written via pandas → registered as external table 'products_pushdown'

Sanity check — plan must show 'FileScan parquet', not 'LocalTableScan':
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet spark_catalog.default.products_pushdown (1)


(1) Scan parquet spark_catalog.default.products_pushdown
Output [2]: [product_id#61, price#64]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/krivg/spark-bq/data/products_pushdown]
ReadSchema: struct<product_id:int,price:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [product_id#61, price#64]




In [5]:
# ── Case 1: Predicate pushdown FIRES ─────────────────────────────────────────
# Filter written AFTER a select — Catalyst pushes it below the Project
# because 'price' exists in the Parquet source.
fires = (
    products_pq
    .select("product_id", "product_name", "category", "price")
    .filter(F.col("price") > 100)
)

print("CASE 1 — filter pushdown FIRES (filter on source column, Parquet source):")
print()
print("Analyzed plan : Filter is ABOVE Project  ← where we wrote it")
print("Optimized plan: Filter is BELOW Project  ← Catalyst moved it (predicate pushdown)")
print("Physical plan : filter appears as PushedFilters on FileScan ← pushed to file reader")
print()
fires.explain(mode="extended")

CASE 1 — filter pushdown FIRES (filter on source column, Parquet source):

Analyzed plan : Filter is ABOVE Project  ← where we wrote it
Optimized plan: Filter is BELOW Project  ← Catalyst moved it (predicate pushdown)
Physical plan : filter appears as PushedFilters on FileScan ← pushed to file reader

== Parsed Logical Plan ==
'Filter ('price > 100)
+- Project [product_id#61, product_name#62, category#63, price#64]
   +- SubqueryAlias spark_catalog.default.products_pushdown
      +- Relation spark_catalog.default.products_pushdown[product_id#61,product_name#62,category#63,price#64,stock#65,description#66,sku#67] parquet

== Analyzed Logical Plan ==
product_id: int, product_name: string, category: string, price: double
Filter (price#64 > cast(100 as double))
+- Project [product_id#61, product_name#62, category#63, price#64]
   +- SubqueryAlias spark_catalog.default.products_pushdown
      +- Relation spark_catalog.default.products_pushdown[product_id#61,product_name#62,category#63,price

***Parsed Logical Plan — "what you wrote, unresolved"***

The filter is above the project — exactly where you wrote it in Python `(.select().filter())`. Column references like 'price are still strings; no types yet.

***Analyzed Logical Plan — "names resolved, types checked"***

Two things happened:

1. 'price became price#64: double — resolved against the Parquet schema, given an internal ID (#64)
2. 100 became cast(100 as double) — Catalyst matched the literal's type to the column's type

***Optimized Logical Plan — "three rules fired"***

Three things changed vs Analyzed:

1. Predicate pushdown: Filter moved below Project. Catalyst proved price exists in the Parquet source, so rows can be discarded before the column selection runs.

2. isnotnull injection: Catalyst added isnotnull(price#64) automatically. SQL semantics say NULL > 100 is NULL (not false), so NULL rows would pass the filter incorrectly. Catalyst inserts the null check to be safe — and it's free because the Parquet reader can skip null values at the same time.

3. Constant folding: cast(100 as double) became 100.0 — evaluated once at plan time, never again per row.

Also notice: SubqueryAlias disappeared. Catalyst flattened the catalog alias away — it's not needed in execution.

***Physical Plan — "how it actually runs"***


Four things to read here:

1. FileScan with ReadSchema (column pruning): Only 4 of 7 columns are listed — stock, description, sku are absent. The Parquet reader will only seek those 4 column byte ranges on disk.

2. PushedFilters: The filter was handed to the Parquet reader itself. The reader skips entire row groups where all price values are ≤ 100 — before Spark even constructs a partition object.

3. DataFilters: The same filter also appears here as a safety net applied by Spark after the Parquet reader runs (in case the row-group-level pushdown passed a partial row group).

4. ColumnarToRow: The vectorized Parquet reader returns columnar Arrow batches. This operator converts them to row format for the upstream Filter operator. The *(1) wrapper around both means they share a single WholeStageCodegen boundary — compiled into one tight loop.


Filter on a derived column can still allow pushdown if Catalyst can algebraically substitute the expression back to a source column. Between the Analyzed and Optimized plans, Catalyst applies **expression substitution**: it rewrites the filter in terms of the source column. Between the Optimized and Physical plans, it can go further with **arithmetic simplification**:

```
price_with_tax > 120   →   (price * 1.2) > 120.0   →   price > 100.0
```

It substituted the alias definition, then divided both sides of the inequality by `1.2` (a non-zero constant), yielding a simpler filter that the Parquet reader can evaluate directly.

| Expression | Pushable? | Why |
|---|---|---|
| `price * 1.2` | ✅ Yes | Simple arithmetic — Catalyst can substitute and simplify |
| `F.concat(col, lit("x"))` | ✅ Partial | `IsNotNull` pushed; the full expression stays in Spark |
| `F.rand()` | ❌ No | Non-deterministic — result changes each call, cannot be moved |
| `F.hash(col)` | ❌ No | Not algebraically invertible |
| window function result | ❌ No | Cannot push past a window boundary |

In [ ]:
# ── Case 2: Predicate pushdown FIRED ────────────────────────────────────────
# Filter on a simple DERIVED column (computed in the select).
# 'price_with_tax' does not exist in the Parquet file — Yet Catalyst CAN push
# the filter past the Project that creates it.
blocked = (
    products_pq
    .select(
        "product_id",
        "product_name",
        (F.col("price") * 1.2).alias("price_with_tax"),
    )
    .filter(F.col("price_with_tax") > 120)
)

print("CASE 2 — filter pushdown FIRED (filter on derived/aliased column):")
print()
print("Optimized plan: Filter is pushed BELOW Project and appears as PushedFilters on FileScan")
print("Physical plan : PushedFilters present on FileScan — filter evaluated by the reader, not Spark")
print()
blocked.explain(mode="extended")

print()
print("─" * 60)
print("WORKAROUND: filter on the original source column instead:")
print("─" * 60)
workaround = (
    products_pq
    .filter(F.col("price") > 100)   # price > 100  ≈  price_with_tax > 120
    .select(
        "product_id",
        "product_name",
        (F.col("price") * 1.2).alias("price_with_tax"),
    )
)
print()
print("Optimized plan: Filter is now BELOW Project and appears as PushedFilters on FileScan")
print()
workaround.explain(mode="extended")

CASE 2 — filter pushdown FIRED (filter on derived/aliased column):

Optimized plan: Filter is pushed BELOW Project and appears as PushedFilters on FileScan
Physical plan : PushedFilters present on FileScan — filter evaluated by the reader, not Spark

== Parsed Logical Plan ==
'Filter ('price_with_tax > 120)
+- Project [product_id#61, product_name#62, (price#64 * 1.2) AS price_with_tax#90]
   +- SubqueryAlias spark_catalog.default.products_pushdown
      +- Relation spark_catalog.default.products_pushdown[product_id#61,product_name#62,category#63,price#64,stock#65,description#66,sku#67] parquet

== Analyzed Logical Plan ==
product_id: int, product_name: string, price_with_tax: double
Filter (price_with_tax#90 > cast(120 as double))
+- Project [product_id#61, product_name#62, (price#64 * 1.2) AS price_with_tax#90]
   +- SubqueryAlias spark_catalog.default.products_pushdown
      +- Relation spark_catalog.default.products_pushdown[product_id#61,product_name#62,category#63,price#64,stock#6

In [9]:
# ── Case 3: Predicate pushdown BLOCKED ────────────────────────────────────────
# Filter on a column derived from a non-deterministic expression (rand()).
# Catalyst cannot push this filter past the Project because rand() must be
# evaluated exactly once per row — moving it would change the result.
blocked_real = (
    products_pq
    .select("product_id", "product_name",
            (F.rand() * F.col("price")).alias("random_price"))
    .filter(F.col("random_price") > 100)
)

print("CASE 3 — filter pushdown BLOCKED (non-deterministic expression):")
print()
print("Optimized plan: Filter stays ABOVE Project — rand() cannot be moved")
print("Physical plan : PushedFilters: [] AND DataFilters: [] — filter runs after Project, not at scan")
print()
blocked_real.explain(mode="extended")

CASE 3 — filter pushdown BLOCKED (non-deterministic expression):

Optimized plan: Filter stays ABOVE Project — rand() cannot be moved
Physical plan : PushedFilters: [] AND DataFilters: [] — filter runs after Project, not at scan

== Parsed Logical Plan ==
'Filter ('random_price > 100)
+- Project [product_id#61, product_name#62, (rand(4038843338425115120) * price#64) AS random_price#102]
   +- SubqueryAlias spark_catalog.default.products_pushdown
      +- Relation spark_catalog.default.products_pushdown[product_id#61,product_name#62,category#63,price#64,stock#65,description#66,sku#67] parquet

== Analyzed Logical Plan ==
product_id: int, product_name: string, random_price: double
Filter (random_price#102 > cast(100 as double))
+- Project [product_id#61, product_name#62, (rand(4038843338425115120) * price#64) AS random_price#102]
   +- SubqueryAlias spark_catalog.default.products_pushdown
      +- Relation spark_catalog.default.products_pushdown[product_id#61,product_name#62,category#63,

### What to observe

**Case 1 — pushdown fires (source column, Parquet):**

```
Optimized plan:
  Project [4 cols]              ← Filter moved BELOW Project by Catalyst
  +- Filter (isnotnull(price) AND price > 100.0)
     +- Relation [...] parquet

Physical plan:
  FileScan parquet [4 cols]
    PushedFilters: [IsNotNull(price), GreaterThan(price,100.0)]
    ReadSchema: 4 of 7 columns
```

Three things happened in one step:
1. **Predicate pushdown** — Filter moved below Project in the Optimized plan.
2. **File-reader pushdown** — Filter handed to the Parquet reader as `PushedFilters`; row groups where all `price` values are ≤ 100 are skipped before Spark constructs a single partition.
3. **Column pruning** — `FileScan` lists only 4 of 7 columns; the reader seeks only those byte ranges on disk.

---

**Case 2 — partial pushdown (simple arithmetic on a derived column):**

```
Optimized plan:
  Project [product_id, product_name, (price * 1.2) AS price_with_tax]
  +- Filter (isnotnull(price) AND (price * 1.2) > 120.0)   ← below Project
     +- Relation [...] parquet

Physical plan:
  Filter (isnotnull(price) AND price > 100.0)              ← further simplified
  +- FileScan parquet [...]
       PushedFilters: [IsNotNull(price), GreaterThan(price,100.0)]
```

Catalyst did **not** block here — it substituted the alias definition into the filter (`price_with_tax → price * 1.2`), then arithmetically simplified `(price * 1.2) > 120` to `price > 100`. The result was pushed all the way to the Parquet reader.

The `PushedFilters` line shows `GreaterThan(price, 100.0)` — the full simplified filter, not just `IsNotNull`. This is the same outcome as Case 1: full row-group skipping at the file reader.

**Key distinction from Case 1:** the `GreaterThan` filter appears in `DataFilters` as `(price * 1.2) > 120.0` (the unsimplified form) but in `PushedFilters` as `GreaterThan(price, 100.0)` (the simplified form). Catalyst simplifies for the reader but retains the logical form for Spark's own filter evaluation as a safety net.

---

**Case 3 — pushdown blocked (non-deterministic expression):**

```
Optimized plan:
  Filter (isnotnull(random_price) AND random_price > 100.0)   ← ABOVE Project
  +- Project [product_id, product_name, (rand() * price) AS random_price]
     +- Relation [...] parquet

Physical plan:
  Filter (isnotnull(random_price) AND random_price > 100.0)
  +- Project [... (rand() * price) AS random_price]
     +- FileScan parquet [...]
          DataFilters: []
          PushedFilters: []
```

Three things to notice:

1. **Filter stays above Project in the Optimized plan** — Catalyst cannot substitute `rand() * price` back to a source column because `rand()` produces a different value on every call. Moving the filter would change which rows survive.

2. **`PushedFilters: []` AND `DataFilters: []`** — the FileScan has no filters at all. The full dataset is read from disk, the Project computes `random_price` for every row, and only then does the Filter discard rows. This is the most expensive execution path.

3. **Column pruning still fires independently** — `ReadSchema` lists only `product_id, product_name, price` (3 of 7 columns). Column pruning does not depend on filter pushdown; Catalyst applies it separately.

**Why `DataFilters` is also empty:** `DataFilters` are filters Spark applies during the scan phase (after the Parquet reader, before passing rows upstream). Because the filter references `random_price`, which only exists after the Project runs, Spark cannot apply it during the scan at all — it must run as a separate operator above the Project.

**Interview answer:** *"Predicate pushdown moves filters to execute as early as possible. With Parquet, it goes all the way to the file reader — visible as `PushedFilters` in the physical plan. It is blocked when the filter cannot be expressed in terms of source columns: non-deterministic functions like `rand()` cannot be moved because re-evaluating them would change the result."*

### 2b. Column pruning

**The rule:** if a `Project` (select) does not reference a column from the scan, remove that column from the plan entirely — Spark will not read it.

**Why it matters for Parquet:**
Parquet is a columnar format. Each column is stored in its own byte range. Spark passes the pruned column list to the Parquet reader, which seeks only to those byte ranges. A query selecting 3 of 100 columns reads ~3 % of the data.

**In `explain()` output:** the `LocalTableScan` (or `FileScan parquet`) node's `Output` list shows only the columns Spark will actually read.

In [14]:
# products_pq has 7 columns in Parquet. Select only 2 — verify column pruning in the plan.
# Using the Parquet source so ReadSchema shows the actual byte-range skipping at the file reader.
narrow = products_pq.select("product_id", "price")
wide   = products_pq.select("*")   # all 7 columns

print("NARROW select (2 of 7 columns) — Parquet source:")
narrow.explain(mode="extended")

print()
print("WIDE select (all 7 columns) — Parquet source:")
wide.explain(mode="extended")

print()
print("KEY: compare ReadSchema on the FileScan node.")
print("Narrow: ReadSchema lists only product_id and price — reader skips the other 5 column byte ranges.")
print("Wide  : ReadSchema lists all 7 columns — reader must seek every column byte range.")

NARROW select (2 of 7 columns) — Parquet source:
== Parsed Logical Plan ==
'Project ['product_id, 'price]
+- SubqueryAlias spark_catalog.default.products_pushdown
   +- Relation spark_catalog.default.products_pushdown[product_id#61,product_name#62,category#63,price#64,stock#65,description#66,sku#67] parquet

== Analyzed Logical Plan ==
product_id: int, price: double
Project [product_id#61, price#64]
+- SubqueryAlias spark_catalog.default.products_pushdown
   +- Relation spark_catalog.default.products_pushdown[product_id#61,product_name#62,category#63,price#64,stock#65,description#66,sku#67] parquet

== Optimized Logical Plan ==
Project [product_id#61, price#64]
+- Relation spark_catalog.default.products_pushdown[product_id#61,product_name#62,category#63,price#64,stock#65,description#66,sku#67] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet spark_catalog.default.products_pushdown[product_id#61,price#64] Batched: true, DataFilters: [], Format: Parquet, Location: InMe

### What to observe

**Narrow select — `ReadSchema: struct<product_id:int,price:double>`**
Only 2 of the 7 columns appear in `ReadSchema`. The Parquet reader physically seeks only the byte ranges for `product_id` and `price` — the other 5 columns (`product_name`, `category`, `stock`, `description`, `sku`) are never read from disk.

**Wide select — `ReadSchema` lists all 7 columns**
Every column byte range is read. Even columns that are never used downstream add I/O and deserialization cost.

**Why `products_pq` (Parquet) and not `products` (in-memory):**
`products` was created with `createDataFrame(list)`, which with Arrow disabled goes through `sc.parallelize()` and produces a `Scan ExistingRDD` in the physical plan. All 7 columns are already in memory from when `sc.parallelize()` ran — column pruning still fires logically (the `Project` node above the scan lists only the selected columns), but no bytes are saved because the RDD was already fully materialized.

With a Parquet source (`products_pq`), the column list in `ReadSchema` is the instruction passed to the file reader — columns absent from it are never read. That is where column pruning delivers real I/O savings.

**Rule of thumb:** column pruning's biggest gains are on wide columnar sources (Parquet, ORC, Delta Lake). On in-memory RDDs or row-format sources (CSV, JSON), Spark prunes logically but there is no I/O saving.

### 2c. Constant folding & boolean simplification

**Constant folding:** any expression whose value can be computed at plan time — before any row is processed — is evaluated once and replaced by its literal value.

```python
df.filter(F.lit(1) + F.lit(1) == F.lit(2))   # always True → row never discarded
df.filter(F.lit(1) + F.lit(1) == F.lit(3))   # always False → entire scan eliminated
```

**Boolean simplification:** `true AND x` → `x`; `x OR true` → `true`; `false AND x` → `false` (eliminates the scan).

These rules are low-level but they explain why Catalyst can eliminate an entire branch of a conditional query without seeing any data.

In [15]:
# ── Constant folding: always-false filter eliminates the scan ─────────────────
always_false = products.filter(F.lit(1) == F.lit(2))   # 1 == 2 → false

print("ALWAYS-FALSE filter — optimized plan should show no scan:")
always_false.explain(mode="extended")

print()
# ── Constant folding: always-true filter is removed entirely ──────────────────
always_true = products.filter(F.lit(1) == F.lit(1))    # 1 == 1 → true
print("ALWAYS-TRUE filter — optimized plan should have no Filter node:")
always_true.explain(mode="extended")

ALWAYS-FALSE filter — optimized plan should show no scan:
== Parsed Logical Plan ==
Filter (1 = 2)
+- LogicalRDD [product_id#0, product_name#1, category#2, price#3, stock#4, description#5, sku#6], false

== Analyzed Logical Plan ==
product_id: int, product_name: string, category: string, price: double, stock: int, description: string, sku: string
Filter (1 = 2)
+- LogicalRDD [product_id#0, product_name#1, category#2, price#3, stock#4, description#5, sku#6], false

== Optimized Logical Plan ==
LocalRelation <empty>, [product_id#0, product_name#1, category#2, price#3, stock#4, description#5, sku#6]

== Physical Plan ==
LocalTableScan <empty>, [product_id#0, product_name#1, category#2, price#3, stock#4, description#5, sku#6]


ALWAYS-TRUE filter — optimized plan should have no Filter node:
== Parsed Logical Plan ==
Filter (1 = 1)
+- LogicalRDD [product_id#0, product_name#1, category#2, price#3, stock#4, description#5, sku#6], false

== Analyzed Logical Plan ==
product_id: int, product_nam

### What to observe

**Always-false:** the Optimized Logical Plan shows `LocalRelation []` — an empty relation with no columns. Catalyst eliminated the scan entirely. No partitions read, no tasks scheduled.

**Always-true:** the `Filter` node disappears from the Optimized plan — Catalyst removed a predicate that is always satisfied. The `LocalTableScan` proceeds without any filter evaluation overhead.

**Production relevance:** parameterized queries where a date range is `start_date = end_date` accidentally become always-false filters and return zero rows — but finish instantly with no I/O. This is why an apparently correct query returning zero results can have a suspiciously short duration.

## 3. Cost-based optimization (CBO)

RBO rewrites the plan structure using rules that are always correct, regardless of data size. CBO goes further: it uses **table and column statistics** to estimate the cost (rows, bytes) of each plan alternative and picks the cheapest one.

**What CBO can do that RBO cannot:**
- **Join order selection:** when joining three or more tables, the order matters. CBO uses row-count estimates to put the smallest intermediate result first.
- **Join strategy selection:** CBO knows estimated table sizes and can choose BHJ vs SMJ based on actual byte estimates rather than only a threshold comparison.

**What CBO requires:** statistics collected on the tables via `ANALYZE TABLE ... COMPUTE STATISTICS`.

**When statistics are absent:** Catalyst falls back to RBO — it uses default assumptions (default output row fractions, no column histograms). This is why a query on a freshly-loaded table sometimes gets a worse plan than an equivalent one on an analyzed table.

**Configuration:**
```python
spark.conf.set("spark.sql.cbo.enabled", "true")                   # enable CBO (default: true in Spark 3+)
spark.conf.set("spark.sql.statistics.histogram.enabled", "true")  # column histograms for better selectivity
```

In [16]:
for name, df in [("orders_cbo", orders), ("customers_cbo", customers), ("products_cbo", products)]:
    path = Path().absolute().parent / "data" / name
    path.mkdir(parents=True, exist_ok=True)
    df.toPandas().to_parquet(str(path / "part-0.parquet"), index=False)
    spark.sql(f"DROP TABLE IF EXISTS {name}")
    spark.sql(f"CREATE TABLE {name} USING PARQUET LOCATION '{path.as_uri()}'")

print("External tables created: orders_cbo, customers_cbo, products_cbo")

orders_t    = spark.table("orders_cbo")
customers_t = spark.table("customers_cbo")
products_t  = spark.table("products_cbo")

External tables created: orders_cbo, customers_cbo, products_cbo


In [21]:
# Three-way join BEFORE running ANALYZE — Catalyst has no size information
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force non-broadcast joins
spark.conf.set("spark.sql.cbo.joinReorder.enabled", "true")

three_way_no_stats = (
    orders_t
    .join(customers_t, "customer_id")
    .join(products_t, "product_id")
    .groupBy("category")
    .agg(F.sum("amount").alias("total"))
)

print("THREE-WAY JOIN — no statistics (Catalyst uses defaults):")
three_way_no_stats.explain(mode="cost")
print()
print("Note the join order — without stats, Catalyst picks arbitrary order.")
print("Look for which table appears as the left/right child of each join node.")

THREE-WAY JOIN — no statistics (Catalyst uses defaults):
== Optimized Logical Plan ==
Aggregate [category#167], [category#167, sum(amount#154) AS total#21301], Statistics(sizeInBytes=156.4 GiB)
+- Project [amount#154, category#167], Statistics(sizeInBytes=156.4 GiB)
   +- Join Inner, (product_id#153 = product_id#165), Statistics(sizeInBytes=191.1 GiB)
      :- Project [product_id#153, amount#154], Statistics(sizeInBytes=126.8 MiB)
      :  +- Join Inner, (customer_id#152 = customer_id#159), Statistics(sizeInBytes=177.5 MiB)
      :     :- Project [customer_id#152, product_id#153, amount#154], Statistics(sizeInBytes=119.1 KiB)
      :     :  +- Filter (isnotnull(customer_id#152) AND isnotnull(product_id#153)), Statistics(sizeInBytes=138.9 KiB)
      :     :     +- Relation spark_catalog.default.orders_cbo[order_id#151,customer_id#152,product_id#153,amount#154] parquet, Statistics(sizeInBytes=138.9 KiB)
      :     +- Project [customer_id#159], Statistics(sizeInBytes=1526.0 B)
      :   

In [18]:
# Run ANALYZE TABLE to collect row counts, byte sizes, and per-column statistics
spark.sql("ANALYZE TABLE orders_cbo    COMPUTE STATISTICS FOR ALL COLUMNS")
spark.sql("ANALYZE TABLE customers_cbo COMPUTE STATISTICS FOR ALL COLUMNS")
spark.sql("ANALYZE TABLE products_cbo  COMPUTE STATISTICS FOR ALL COLUMNS")
print("Statistics collected.")

# Show what was collected for each table
for tbl in ["orders_cbo", "customers_cbo", "products_cbo"]:
    stats = spark.sql(f"DESCRIBE EXTENDED {tbl}").filter(
        F.col("col_name").isin("Statistics")
    ).collect()
    if stats:
        print(f"  {tbl}: {stats[0]['data_type']}")
    else:
        # DESCRIBE EXTENDED may label it differently
        spark.sql(f"DESCRIBE DETAIL {tbl}").show(1, truncate=False) if False else None
        print(f"  {tbl}: stats stored in catalog (check DESCRIBE EXTENDED {tbl})")

Statistics collected.
  orders_cbo: 142264 bytes, 10000 rows
  customers_cbo: 4579 bytes, 200 rows
  products_cbo: 5021 bytes, 20 rows


In [22]:
# Same three-way join AFTER ANALYZE — CBO now has row counts and byte estimates
three_way_with_stats = (
    spark.table("orders_cbo")
    .join(spark.table("customers_cbo"), "customer_id")
    .join(spark.table("products_cbo"), "product_id")
    .groupBy("category")
    .agg(F.sum("amount").alias("total"))
)

print("THREE-WAY JOIN — WITH statistics (CBO can estimate costs):")
three_way_with_stats.explain(mode="cost")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

print()
print("Compare join order against the no-stats plan.")
print("With stats, CBO should put the smallest table (products: 20 rows) earlier in the join tree.")

THREE-WAY JOIN — WITH statistics (CBO can estimate costs):
== Optimized Logical Plan ==
Aggregate [category#21227], [category#21227, sum(amount#21206) AS total#21351], Statistics(sizeInBytes=132.0 B, rowCount=4)
+- Project [amount#21206, category#21227], Statistics(sizeInBytes=823.2 KiB, rowCount=2.55E+4)
   +- Join Inner, (customer_id#21204 = customer_id#21211), Statistics(sizeInBytes=1022.8 KiB, rowCount=2.55E+4)
      :- Project [customer_id#21204, amount#21206, category#21227], Statistics(sizeInBytes=415.7 KiB, rowCount=1.15E+4)
      :  +- Join Inner, (product_id#21205 = product_id#21225), Statistics(sizeInBytes=505.6 KiB, rowCount=1.15E+4)
      :     :- Project [customer_id#21204, product_id#21205, amount#21206], Statistics(sizeInBytes=234.4 KiB, rowCount=1.00E+4)
      :     :  +- Filter (isnotnull(customer_id#21204) AND isnotnull(product_id#21205)), Statistics(sizeInBytes=273.4 KiB, rowCount=1.00E+4)
      :     :     +- Relation spark_catalog.default.orders_cbo[order_id#21203

### What to observe: CBO before vs after `ANALYZE TABLE`

**Without statistics:**
- Catalyst has no row-count information. It falls back to default size estimates.
- Join order may be arbitrary — the plan tree structure reflects source table order, not size order.
- Size estimates in plan nodes will show large default numbers (often `-1` or unrealistic values).

**With statistics:**
- Catalyst knows `products` = 20 rows, `customers` = 200 rows, `orders` = 10 000 rows.
- CBO reorders joins to minimize intermediate result sizes: join the smallest dimension first, then the larger one.
- Size estimates in the plan nodes reflect actual row counts and byte sizes.

**`explain(mode='cost')` (Spark 3.1+):** prints estimated row counts and data sizes at each node — shows exactly what CBO is working with. Try `three_way_with_stats.explain(mode='cost')` to see the numbers.

**Interview answer:** *"CBO requires statistics — without `ANALYZE TABLE`, Catalyst uses guesses. With statistics, it can choose the cheapest join order and avoid unnecessary shuffles."*

In [20]:
# explain(mode='cost') shows estimated row/byte counts at each node
print("Cost mode — shows estimated rows and bytes at each plan node:")
three_way_with_stats.explain(mode="cost")

Cost mode — shows estimated rows and bytes at each plan node:
== Optimized Logical Plan ==
Aggregate [category#21227], [category#21227, sum(amount#21206) AS total#21265], Statistics(sizeInBytes=132.0 B, rowCount=4)
+- Project [amount#21206, category#21227], Statistics(sizeInBytes=370.8 KiB, rowCount=1.15E+4)
   +- Join Inner, (product_id#21205 = product_id#21225), Statistics(sizeInBytes=460.6 KiB, rowCount=1.15E+4)
      :- Project [product_id#21205, amount#21206], Statistics(sizeInBytes=498.9 KiB, rowCount=2.55E+4)
      :  +- Join Inner, (customer_id#21204 = customer_id#21211), Statistics(sizeInBytes=698.5 KiB, rowCount=2.55E+4)
      :     :- Project [customer_id#21204, product_id#21205, amount#21206], Statistics(sizeInBytes=234.4 KiB, rowCount=1.00E+4)
      :     :  +- Filter (isnotnull(customer_id#21204) AND isnotnull(product_id#21205)), Statistics(sizeInBytes=273.4 KiB, rowCount=1.00E+4)
      :     :     +- Relation spark_catalog.default.orders_cbo[order_id#21203,customer_id#21

## 4. Tungsten execution engine

Catalyst is the logical optimizer. Tungsten is the **physical execution engine** — it determines how operators actually run on JVM hardware. Catalyst produces the plan; Tungsten executes it fast.

Tungsten has three major components:

### 4a. Off-heap memory management
Standard Java objects on the heap have a large overhead: each object has a 16-byte header, plus internal field padding. A 10-character string can occupy 120+ bytes on heap.

Tungsten stores data in **compact binary row format** (`UnsafeRow`) directly in off-heap memory — bypassing the JVM garbage collector. A row with a single integer and double occupies exactly 12 bytes, with no GC overhead.

**Result:** dramatically less GC pressure, smaller memory footprint per row, and cache-line-friendly data access patterns.

### 4b. Cache-aware algorithms
Tungsten implements sort and aggregation algorithms designed to access data in sequential memory order — maximizing CPU cache utilization. On modern hardware, a cache miss costs ~100 CPU cycles; sequential access costs ~4 cycles.

### 4c. WholeStageCodegen
Tungsten generates specialized JVM bytecode for each query at runtime — fusing multiple operators into a single tight loop. This is the most visible Tungsten feature in `explain()` output.

## 5. WholeStageCodegen — operator fusion

**The problem with a volcano model (pre-Tungsten):**
Each operator calls `next()` on its child to get one row at a time. This means a virtual function call for every row crossing every operator boundary. At 10M rows/sec with 5 operators, that is 50M virtual calls per second — expensive.

**WholeStageCodegen (WSC):**
Tungsten generates a single bytecode method that handles `Filter`, `Project`, and `HashAggregate(partial)` all in one tight loop — no virtual calls between them. The `WholeStageCodegen (N)` box in `explain()` marks which operators are fused.

**Which operators break codegen (cross a boundary):**

| Operator | Fused? | Reason |
|---|---|---|
| `Filter` | Yes | Simple predicate, compiled inline |
| `Project` | Yes | Column expressions, compiled inline |
| `HashAggregate (partial)` | Yes | Map-side aggregation, fused with input scan |
| `HashAggregate (final)` | Yes/No | Usually, though may split if complex |
| `SortMergeJoin` | **No** | Iterator-based merge; cannot be fully inlined |
| `BroadcastHashJoin` | **No** | Hash probe requires non-inlinable lookup |
| `Sort` | **No** | Spill-aware sort breaks the single-pass model |
| `Exchange` | **No** | Shuffle boundary — always a codegen boundary |

**How to read it in `explain()`:**
Operators inside the same `WholeStageCodegen (N)` box are compiled together. Crossing a box boundary means a row is materialized and passed to the next operator via a regular function call.

In [53]:
# ── Narrow pipeline: all operators fuse into one WSC boundary ─────────────────

# explain(mode='codegen') shows the generated Java code for each WSC region.
# This is what Tungsten compiles and executes — one tight loop per region.
#
# AQE CAVEAT: when spark.sql.adaptive.enabled=true (our default), the plan is
# wrapped in AdaptiveSparkPlanExec before execution. explain(mode='codegen')
# cannot look inside that wrapper, so it reports "Found 0 WholeStageCodegen
# subtrees" until the query has actually run and AQE marks isFinalPlan=true.
#
# To inspect codegen without running an action, temporarily disable AQE:
spark.conf.set("spark.sql.adaptive.enabled", "false")
narrow_pipeline = (
    orders
    .filter(F.col("amount") > 500)
    .select("order_id", "customer_id", "amount")
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("total"))
)

print("NARROW PIPELINE — filter + project + groupBy:")
print("Expect all ops inside a single WholeStageCodegen boundary (no Exchange)")

# narrow_pipeline_collection = narrow_pipeline.collect()  # force execution to see the actual plan with WSC applied in the UI and the narrow_pipeline.explain(mode="codegen") detects the WholeStageCodegen subtrees. You will create a codegen boundary as the final aggregation will be executed in a separate stage as it requires shuffling the data, so we will see two WSC regions instead of one.
narrow_pipeline.explain(mode="codegen")

NARROW PIPELINE — filter + project + groupBy:
Expect all ops inside a single WholeStageCodegen boundary (no Exchange)
Found 2 WholeStageCodegen subtrees.
== Subtree 1 / 2 (maxMethodCodeSize:248; maxConstantPoolSize:287(0.44% used); numInnerClasses:2) ==
*(1) HashAggregate(keys=[customer_id#21], functions=[partial_sum(amount#23)], output=[customer_id#21, sum#21676])
+- *(1) Project [customer_id#21, amount#23]
   +- *(1) Filter (isnotnull(amount#23) AND (amount#23 > 500.0))
      +- *(1) Scan ExistingRDD[order_id#20,customer_id#21,product_id#22,amount#23]

Generated code:
/* 001 */ public Object generate(Object[] references) {
/* 002 */   return new GeneratedIteratorForCodegenStage1(references);
/* 003 */ }
/* 004 */
/* 005 */ // codegenStageId=1
/* 006 */ final class GeneratedIteratorForCodegenStage1 extends org.apache.spark.sql.execution.BufferedRowIterator {
/* 007 */   private Object[] references;
/* 008 */   private scala.collection.Iterator[] inputs;
/* 009 */   private boolean has

In [54]:
# ── Wide pipeline: Exchange breaks codegen; join also breaks it ───────────────
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force SMJ

wide_pipeline = (
    orders
    .filter(F.col("amount") > 500)
    .join(customers, "customer_id")   # SMJ: Exchange on both sides + Sort
    .groupBy("country_id")
    .agg(F.sum("amount").alias("total"))
)

print("WIDE PIPELINE — filter + SMJ + groupBy (Exchange present):")
print("Expect MULTIPLE WholeStageCodegen regions separated by Exchange and Sort nodes")

# wide_pipeline_collection = wide_pipeline.collect() 
wide_pipeline.explain(mode="codegen")


spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

WIDE PIPELINE — filter + SMJ + groupBy (Exchange present):
Expect MULTIPLE WholeStageCodegen regions separated by Exchange and Sort nodes
Found 6 WholeStageCodegen subtrees.
== Subtree 1 / 6 (maxMethodCodeSize:235; maxConstantPoolSize:126(0.19% used); numInnerClasses:0) ==
*(1) Project [customer_id#21, amount#23]
+- *(1) Filter ((isnotnull(amount#23) AND (amount#23 > 500.0)) AND isnotnull(customer_id#21))
   +- *(1) Scan ExistingRDD[order_id#20,customer_id#21,product_id#22,amount#23]

Generated code:
/* 001 */ public Object generate(Object[] references) {
/* 002 */   return new GeneratedIteratorForCodegenStage1(references);
/* 003 */ }
/* 004 */
/* 005 */ // codegenStageId=1
/* 006 */ final class GeneratedIteratorForCodegenStage1 extends org.apache.spark.sql.execution.BufferedRowIterator {
/* 007 */   private Object[] references;
/* 008 */   private scala.collection.Iterator[] inputs;
/* 009 */   private scala.collection.Iterator rdd_input_0;
/* 010 */   private org.apache.spark.sql.ca

### What to observe in the WSC plans

**Narrow pipeline:**
- `WholeStageCodegen (1)`: scan ` → ` Filter` → `Project` → `HashAggregate(partial)
- Exchange (shuffle boundary) → codegen boundary.
- `WholeStageCodegen (2)`: HashAggregate(Final).

**Wide pipeline:**
- `WholeStageCodegen (1)`: scan + filter on orders side (left).
- Exchange (shuffle boundary) → codegen boundary.
- `WholeStageCodegen (2)`: scan + sort on customers side.
- `WholeStageCodegen (3)`: sort on orders side after shuffle.
- `WholeStageCodegen (4)`: sort on customers side after shuffle.
- `WholeStageCodegen (5)`: SortMergeJoin itself + Project + HashAggregate(Partial)
- Exchange (second shuffle for groupBy) → codegen boundary.
- `WholeStageCodegen (6)`: HashAggregate(Final).

**Interview answer for "which operators break WholeStageCodegen":**
*"Exchange nodes always break WSC because they are shuffle boundaries — different stages. Joins (SMJ, BHJ) break WSC because they require merging two independent row streams, which can't be fully inlined. Sort breaks WSC because it needs to buffer and spill. Everything between breaks — Filter, Project, partial HashAggregate — is fused into one compiled method."*

In [43]:
# explain(mode='codegen') shows the actual generated Java code for each WSC region
# This is what Tungsten compiles and executes — one tight loop per region.
print("Generated codegen for the narrow pipeline (first WSC region):")
print("Note the absence of virtual dispatch — it is a direct while-loop with inlined predicates.")
narrow_pipeline.explain(mode="codegen")

Generated codegen for the narrow pipeline (first WSC region):
Note the absence of virtual dispatch — it is a direct while-loop with inlined predicates.
Found 0 WholeStageCodegen subtrees.



## Spark UI checklist for Catalyst analysis

### SQL tab walkthrough

**Step 1 — Find the job in the SQL tab**
Every action (count, show, collect) registers a SQL execution. Click the description to open the plan.

**Step 2 — Verify predicate pushdown fired**
- Find the `Filter` node in the plan.
- Check its position relative to any `Project` or `Join` node on the same branch.
- `Filter` **below** a `Project` = pushdown fired ✓.
- `Filter` **above** a `Project` = pushdown did not fire — investigate the filter expression.

**Step 3 — Verify column pruning**
- Click on the `LocalTableScan` or `FileScan` node.
- Check the `Output` list — it should contain only the columns referenced downstream.
- If it shows all columns from a wide table, column pruning may not have fired (rare for simple selects; more common with complex views).

**Step 4 — Count WholeStageCodegen regions**
- Each `WholeStageCodegen (N)` box is one compiled loop.
- More boxes = more operator boundaries = more overhead.
- For a pure narrow pipeline (filter + project + agg), expect 1–2 boxes.
- For a join pipeline (SMJ), expect 4–6 boxes.

**Step 5 — Read size estimates (if CBO is enabled)**
- Hover over or click join nodes in the SQL DAG — size estimates appear.
- A size of `-1` or very large defaults = no statistics = CBO had no data to work with.
- Run `ANALYZE TABLE` to fix.

## Interview questions — answer from memory before moving on

### Q1. Explain the Catalyst optimizer. What are its four plan stages?

**A:**
Catalyst is Spark SQL's query optimizer. Every DataFrame or SQL query passes through four stages:
1. **Parsed logical plan:** the SQL/Python is parsed into an AST. Column names are unresolved (`'col`).
2. **Analyzed logical plan:** the Analyzer resolves names against the catalog and assigns attribute IDs and types. This is where `AnalysisException` is thrown for unknown columns.
3. **Optimized logical plan:** the Optimizer applies rule batches (RBO) and optionally cost-based rewrites (CBO): predicate pushdown, column pruning, constant folding, subquery elimination, join reordering.
4. **Physical plan:** the Planner selects concrete operators — `BroadcastHashJoin` vs `SortMergeJoin`, `Exchange` nodes, `WholeStageCodegen` boundaries.

Use `explain(mode='extended')` to see all four stages at once. The most informative comparison is Analyzed vs Optimized — that delta shows exactly what Catalyst changed.

---

### Q2. What is predicate pushdown? Give a concrete example of where it does and does not fire.

**A:**
Predicate pushdown moves a `Filter` node as close to the data source as possible — before projections, before joins, before aggregations. This reduces the number of rows that flow through subsequent expensive operators.

**Fires:** `df.select("a", "b").filter(col("a") > 10)` — Catalyst moves the filter below the project because `a` exists in the source. The optimized plan shows `Filter` below `Project`.


**May fire:** Filter on a derived column can still allow pushdown if Catalyst can algebraically substitute the expression back to a source column. Between the Analyzed and Optimized plans, Catalyst applies **expression substitution**: it rewrites the filter in terms of the source column. Between the Optimized and Physical plans, it can go further with **arithmetic simplification**:

```
price_with_tax > 120   →   (price * 1.2) > 120.0   →   price > 100.0
```

**Does not fire:** Filter stays above Project in the Optimized plan — Catalyst cannot substitute `rand() * price` back to a source column because `rand()` produces a different value on every call. Moving the filter would change which rows survive.

---

### Q3. What is column pruning? Why does it matter for columnar file formats like Parquet?

**A:**
Column pruning removes unreferenced columns from the scan plan node. If a query selects 3 of 100 columns, Catalyst generates a plan that reads only those 3 columns.

For **Parquet**, this is a major I/O win: Parquet stores each column in a separate byte range (column chunk). Spark passes the pruned column list to the Parquet reader, which seeks only to those byte ranges — skipping the other 97 columns entirely. A query on a 1 TB table selecting 3 columns might read only ~30 GB.

For **CSV/JSON** (row formats), every row must be read regardless — column pruning reduces CPU parsing work but not I/O.

Verify in `explain()`: the `FileScan parquet` or `LocalTableScan` node's `Output` list should show only the referenced columns.

---

### Q4. What is the difference between rule-based and cost-based optimization in Catalyst?

**A:**
**Rule-based optimization (RBO):** applies algebraic rewrite rules unconditionally — the rules are always correct regardless of data size. Examples: predicate pushdown, column pruning, constant folding. No statistics required.

**Cost-based optimization (CBO):** estimates the cost (rows, bytes) of alternative plan shapes and selects the cheapest one. Examples: join order selection when three or more tables are joined, join strategy selection based on actual size estimates. **Requires statistics** — collected via `ANALYZE TABLE ... COMPUTE STATISTICS FOR ALL COLUMNS`.

Without statistics, CBO falls back to RBO-only behavior with default size guesses. With stale statistics, CBO can make worse decisions than RBO defaults — always re-run `ANALYZE TABLE` after significant data changes.

---

### Q5. What statistics does Spark use for CBO? How are they collected?

**A:**
Spark collects two levels of statistics:

**Table-level:** row count and total byte size. Used to compare table sizes for join strategy and join order decisions.

**Column-level** (when `FOR ALL COLUMNS` or `FOR COLUMNS col1, col2` is specified):
- min/max values
- null count
- number of distinct values (NDV)
- column histogram (when `spark.sql.statistics.histogram.enabled=true`) — bucket-based distribution for better selectivity estimates

**Collection:**
```sql
ANALYZE TABLE my_table COMPUTE STATISTICS;                    -- table-level only
ANALYZE TABLE my_table COMPUTE STATISTICS FOR ALL COLUMNS;   -- + column stats
ANALYZE TABLE my_table COMPUTE STATISTICS FOR COLUMNS col1, col2;  -- specific columns
```
Statistics are stored in the Hive metastore (or Spark's internal catalog). They are **not updated automatically** — re-run after large writes.

---

### Q6. What is Tungsten? How does it differ from standard JVM memory management?

**A:**
Tungsten is Spark's physical execution engine, introduced in Spark 1.5. It addresses three JVM limitations:

1. **Off-heap binary storage (`UnsafeRow`):** instead of storing rows as Java objects (with ~12-byte header overhead per object, plus field padding), Tungsten stores rows in a compact binary format off-heap. Avoids GC pressure — the GC cannot see off-heap memory.

2. **Cache-aware algorithms:** sort and hash-based aggregations are implemented to access memory sequentially, maximizing L1/L2 cache hit rates. A cache miss costs ~100 CPU cycles vs ~4 cycles for a cache hit.

3. **WholeStageCodegen:** generates JVM bytecode at runtime that fuses multiple operators into one tight loop, eliminating the virtual function call overhead of the traditional volcano iterator model.

**Standard JVM vs Tungsten:** the JVM GC must scan all live objects on the heap — with millions of row objects, GC pauses are frequent. Tungsten's off-heap approach means no row objects on the JVM heap, so GC only runs for operator-level objects (much fewer), dramatically reducing pause time.

---

### Q7. Which operators break `WholeStageCodegen` fusion, and why?

**A:**
**Breaks fusion:**
- `Exchange` (shuffle) — always a stage boundary; rows are serialized to disk between stages.
- `SortMergeJoin` — merges two sorted iterators in lock-step; can't be fully inlined into one loop.
- `BroadcastHashJoin` — hash probe involves a hash map lookup that breaks the straight-line code model.
- `Sort` — sort requires buffering all input, potentially spilling to disk; breaks the streaming model.

**Does not break fusion (can be inlined):**
- `Filter` — a simple predicate check, compiled as an `if` statement in the generated code.
- `Project` — column expressions compiled as direct field accesses.
- `HashAggregate (partial)` — map-side aggregation, fused with the input scan.

**Practical impact:** each codegen boundary materializes rows and passes them via a standard function call — slow but necessary. Minimizing codegen boundaries (fewer joins, avoid unnecessary sorts) reduces overhead. A narrow pipeline (filter → project → agg with no shuffle) is one codegen region = maximum throughput.

## Key takeaways — cheat sheet

**The four stages in one sentence each:**
- **Parsed:** AST with unresolved names.
- **Analyzed:** names resolved, types checked — errors throw here.
- **Optimized:** RBO rules fired (pushdown, pruning, folding); CBO join-order if stats available.
- **Physical:** operator selection (BHJ/SMJ), Exchange nodes, WSC boundaries.

**Predicate pushdown quick check:** find the `Filter` in `explain(mode='formatted')` — is it below or above the nearest `Project`/`Join`/`Exchange`? Below = good. Above = investigate.

**Column pruning quick check:** `LocalTableScan` or `FileScan parquet` `Output` list — does it list only the columns you selected? If not, a view or derived column is preventing pruning.

**CBO requires ANALYZE TABLE.** After any large table write, re-run `ANALYZE TABLE ... COMPUTE STATISTICS FOR ALL COLUMNS`.

**WholeStageCodegen boundaries = operator crossing cost.** Each Exchange, each join node, each Sort — one more boundary. Narrow pipelines have the fewest boundaries and the highest throughput.

**Tungsten in one sentence:** compact binary rows off-heap + cache-aware algorithms + JIT-compiled operators = no GC pressure, minimal virtual dispatch, CPU-efficient execution.

**Next notebook:** `06_aqe_deep_dive.ipynb` — AQE is Catalyst running *at runtime*, after each shuffle stage, with access to actual data statistics. The three AQE features (partition coalescing, SMJ→BHJ conversion, skew join splitting) all build directly on the Catalyst plan stages you learned here.

In [ ]:
# Drop all managed tables created in this notebook
for tbl in ["orders_cbo", "customers_cbo", "products_cbo", "products_pushdown"]:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {tbl}")
        print(f"Dropped {tbl}")
    except Exception as e:
        print(f"Cleanup warning ({tbl}): {e}")

print("\nNotebook complete. Spark UI still available at http://localhost:4040")
print("Run spark.stop() to shut down the session.")